In [1]:
import gc
import re
import time
import sys
 
import numpy as np
import pandas as pd
import requests
import scanpy as sc
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import to_rgb
from matplotlib.patches import Patch
from matplotlib.cm import ScalarMappable

In [2]:
print(
    f"Python {sys.version.split()[0]} | NumPy {np.__version__} | Pandas {pd.__version__} | "
    f"Scanpy {sc.__version__} | AnnData {ad.__version__} | "
    f"Seaborn {sns.__version__} | Matplotlib {plt.matplotlib.__version__}"
)

Python 3.13.5 | NumPy 2.2.6 | Pandas 2.3.2 | Scanpy 1.11.4 | AnnData 0.12.2 | Seaborn 0.13.2 | Matplotlib 3.10.5


In [3]:
# Paths
INPUT_DIR = 'Figure4_panelb/'

MS_TSV = INPUT_DIR + 'Experiment1_Report_Protein_Gene Quant (Pivot).tsv'

RAW_SC_PATHS = {
    "mes": ("scObjects_annotated/mesenchyme_large1_cleaned_newlabels.h5ad", "CAF_cluster_labels"),
    "end": ("scObjects_annotated/endothelial_large1_clean_newlabels.h5ad", "Endothelial_cluster_labels"),
    "hem": ("scObjects_annotated/immune_lognorm.h5ad", "Cell Type"),
    "ep":  ("scObjects_annotated/ductal_malignant_lognorm_20270728.h5ad", "cell_type"),
}

OUT_DIR = "output/"
SC_OUT_H5AD = OUT_DIR + "PDAC_scRNAseq_extended.h5ad"  
PDAC_GENESETS_CSV = INPUT_DIR + "PDAC_genesets.csv"

In [4]:
MALIGNANT_CT = ['edTFhigh_Classical', 'edTFlow_Classical', 'edTFnull_Classical', 'S100A4+_Classical-like', 'Basal']
IMMUNE_CT = ['CD8-positive  alpha-beta T cell', 'CD4-positive  alpha-beta T cell', 'T regs', 'NK cells',
             'B lymphocytes', 'Macrophages', 'mDC', 'Monocytes', 'pDC', 'MDSCs', 'Mast cells', 'RBCs']
STROMA_CT = ['CD36+ capillaries', 'capillaries', 'veins', 'ESM1+ Tip cells', 'arteries',
             #'CCL21+ PDPN+ lymphatics', 
             'myCAF 2', 'myCAF 1', 'CD105+ CAFs', 'POSTN+ CD105+ CAFs',
             'iCAF 1', 'iCAF 2', #'CXCL13+ CLU+ CD105+ CAFs', 
             'CCL21+ Reticular CAFs',
              'Schwann cells',
             'Vascular smooth muscle cells', 'CLU+ Vascular smooth muscle cells', 'Pericytes']
BROAD_ME_MAP = {"Malignant": ['CH', 'CL', 'NB', 'SH', 'H', 'B'], "Immune": ['I', 'M'],
                "Stroma": ['S', 'E', 'CM', 'D'], "IS": ['IS'], "CCF": ['CCF']}
BROAD_ME_TO_GROUP = {code: g for g, lst in BROAD_ME_MAP.items() for code in lst}

CELL_TYPE_ORDER = [
    'tumor-associated normal','edTFhigh_Classical','edTFlow_Classical','edTFnull_Classical',
    'S100A4+_Classical-like','Hybrid','Basal',
    'Mast cells','RBCs',
    'MDSCs','Monocytes','Macrophages','mDC','pDC',
    'CD8-positive  alpha-beta T cell','CD4-positive  alpha-beta T cell','T regs',
    'NK cells','B lymphocytes',
    'CD36+ capillaries','capillaries','veins','ESM1+ Tip cells','arteries',
    #'CCL21+ PDPN+ lymphatics',
    'myCAF 2','myCAF 1','CD105+ CAFs','POSTN+ CD105+ CAFs',
    'iCAF 1','iCAF 2',#'CXCL13+ CLU+ CD105+ CAFs',
    'CCL21+ Reticular CAFs',
    'Schwann cells',
    'Vascular smooth muscle cells','CLU+ Vascular smooth muscle cells','Pericytes'
]

ME_ORDER = ['CH', 'CL', 'SH', 'NB', 'H', 'B', 'M', 'I', 'IS', 'S', 'VNC', 'E', 'CCF', 'CM', 'D']
ME_TYPES_CHECK = ['CH', 'CL', 'SH', 'NB', 'H', 'B']

GROUPS = {
    "Malignant": ['CH', 'CL', 'NB', 'SH', 'H', 'B'],
    "Immune": ['I', 'M'],
    "Stroma": ['CCF', 'S', 'E', 'CM', 'D'],
    "IS": ['IS'],
}
TYPE_TO_GROUP = {me: g for g, members in GROUPS.items() for me in members}

INVESTIGATIONAL_PDAC_TARGETS = {
    "KRAS": "Daraxonrasib (RMC-6236) - Phase 3, RAS(ON) inhibitor, positive OS data ASCO 2026",
    "PTPN11": "SHP2 inhibitors (e.g. RMC-4630, TNO155) - combination partner with KRAS inhibitors",
    "WEE1": "Adavosertib - DNA damage response, PDAC combination trials",
    "CHEK1": "Prexasertib - DNA damage response, PDAC combination trials",
    "ATR": "Ceralasertib - DNA damage response, PDAC combination trials",
    "CD40": "CD40 agonists (e.g. sotigalimab) - immune/stromal reprogramming",
    "CCR2": "CCR2 antagonists (e.g. PF-04136309, BMS-813160) - TAM/stromal modulation",
    "CXCR4": "CXCR4 antagonists (e.g. motixafortide) - stromal modulation",
    "CLDN18": "Zolbetuximab and related agents - explored in PDAC beyond gastric",
    "MSLN": "Anetumab ravtansine, mesothelin CAR-T - PDAC-relevant antigen",
    "FAK": "Defactinib - focal adhesion kinase, PDAC stromal target (gene: PTK2)",
    "PTK2": "Defactinib - focal adhesion kinase, PDAC stromal target",
    "STAT3": "p-STAT3 - IL-6/JAK-STAT3 axis",
    "CEACAM5": "CEACAM5 (CEA) ADC target - e.g. tusamitamab ravtansine",
    "CEACAM6": "CEACAM6, linked to gemcitabine resistance/invasion",
}

NCI_CANCER_TYPE_URLS = {
    "breast": "https://www.cancer.gov/about-cancer/treatment/drugs/breast",
    "pancreatic": "https://www.cancer.gov/about-cancer/treatment/drugs/pancreatic",
    "anal": "https://www.cancer.gov/about-cancer/treatment/drugs/anal",
    "colorectal": "https://www.cancer.gov/about-cancer/treatment/drugs/colorectal",
    "esophageal": "https://www.cancer.gov/about-cancer/treatment/drugs/esophageal",
    "gist": "https://www.cancer.gov/about-cancer/treatment/drugs/gist",
    "liver": "https://www.cancer.gov/about-cancer/treatment/drugs/liver",
    "stomach": "https://www.cancer.gov/about-cancer/treatment/drugs/stomach",
    "ovarian": "https://www.cancer.gov/about-cancer/treatment/drugs/ovarian",
    "lung": "https://www.cancer.gov/about-cancer/treatment/drugs/lung",
    "cervical": "https://www.cancer.gov/about-cancer/treatment/drugs/cervical",
}

DGIDB_URL = "https://dgidb.org/api/graphql"

In [5]:
# Mass Spec data: Load, filter and impute

def parse_meta(col):
    col = col.split("]")[-1].strip()
    parts = col.split("_")
    if parts[1] == "Jackson":
        batch = "_".join(parts[1:4])
        sample_type = parts[4]
        if batch == "Jackson_NBCC447_Ferris":
            patient_id = "222596"
        elif parts[5] == "DIA":
            patient_id = "67668"
        else:
            patient_id = parts[5]
    elif parts[1] == "SPORT8" and parts[2] == "Jackson":
        batch = "SPORT8_Jackson"
        sample_type = parts[3]
        plate_idx = next(i for i, p in enumerate(parts) if re.match(r"S\d+-[A-Z]\d+", p))
        patient_id = "_".join(parts[4:plate_idx])
    elif parts[1] == "SPORT8":
        batch = "SPORT8"
        sample_type = parts[2]
        plate_idx = next(i for i, p in enumerate(parts) if re.match(r"S\d+-[A-Z]\d+", p))
        patient_id = "_".join(parts[3:plate_idx])
    else:
        return pd.Series([np.nan, np.nan, np.nan])
    return pd.Series([batch, sample_type, patient_id])


def impute_minprob(df, q=0.1, shift=1.8, width=0.3, seed=42):
    np.random.seed(seed)
    df_imp = df.copy()
    for col in df_imp.columns:
        x = df_imp[col]
        observed = x.dropna()
        if len(observed) < 3:
            continue
        low_vals = observed.nsmallest(max(3, int(len(observed) * q)))
        mu, sigma = low_vals.mean(), low_vals.std()
        imp_mean, imp_sd = mu - shift * sigma, sigma * width
        n_missing = x.isna().sum()
        df_imp.loc[x.isna(), col] = np.random.normal(loc=imp_mean, scale=imp_sd, size=n_missing)
    return df_imp


def load_mass_spec():
    """Returns (df_unimputed_log2, df_filtered_imputed, meta_filtered)."""
    df = pd.read_csv(MS_TSV, sep='\t', index_col=1)
    df = df[~df['PG.ProteinGroups'].str.contains("Cont", case=False, na=False)]

    meta = df.columns[1:].to_series().apply(parse_meta)
    meta.columns = ["batch", "sample_type", "patient_id"]
    meta["non_nan_protein_count"] = df.iloc[:, 1:].notna().sum(axis=0)
    meta["ME_Type"] = meta["sample_type"].str.replace(r"\d+$", "", regex=True)

    # renaming map from Ferris Nowlan
    rename_map = {
        "T1_67329": "E4_67329",
        "T1_631322": "E4_631322",
        "T2_631322": "S4_631322",
        "T4_631322": "E5_631322",
        "T1_63648": "S4_63648",
        "T2_63648": "S5_63648",
        "T3_63648": "CCF4_63648",
        "T3_299324": "CCF1_299324",
        "T3_1230064": "E4_1230064",
        # pilot
        "T1_222596": "E5_222596",
        "T2_222596": "S4_222596",
    }
    combined = meta["sample_type"] + "_" + meta["patient_id"]
    combined = combined.replace(rename_map)
    meta["sample_type"] = combined.str.split("_").str[0]
    meta["patient_id"] = combined.str.split("_").str[1]

    mask = ~meta["batch"].isin(["SPORT8", "SPORT8_Jackson"]) & (meta["ME_Type"] == "H")
    meta.loc[mask, "ME_Type"] = "CL"
    me_replacements = {
        "C": "CH",
        "LS": "IS",
        "VCN": "VNC",
        "CD": "CCF",
        "NB3of": "NB",
        "LV": "I",
        "L": "I",
        "Col1a": "CM",
        "TNC": "E",
        "Nonmalignant": "EX",
        "Immunosuppressed": "M",
        "MDSCs": "M",
        "Immune": "I",
        "ImmuneStroma": "IS",
        "T": "CM",
        "NEO": "D"
    }
    meta["ME_Type"] = meta["ME_Type"].replace(me_replacements)

    meta['patient_ID_num'] = meta['patient_id'].astype(str).str.extract(r'(\d+)')
    meta['patient_ID_num'] = meta['patient_ID_num'].replace({'1263468': '1262468', '604165': '60416'})
    meta['sample_name'] = meta.index
    meta = meta.set_index('sample_name')

    mask = df.iloc[:, 1:].notna().sum(axis=0) > 1500
    kept_samples = df.columns[1:][mask]
    df_filtered = df[kept_samples]
    meta_filtered = meta.loc[kept_samples]

    meta_filtered = meta_filtered[meta_filtered["ME_Type"] != "VNC"]
    counts = meta_filtered["ME_Type"].value_counts()
    valid_types = counts[counts >= 7].index
    meta_filtered = meta_filtered[meta_filtered["ME_Type"].isin(valid_types)]
    df_filtered = df_filtered[meta_filtered.index]

    df_unimputed = np.log2(df_filtered)
    df_filtered_imputed = impute_minprob(df_unimputed)

    gc.collect()
    return df_unimputed, df_filtered_imputed, meta_filtered

In [6]:
# Gene filter + Wilcoxon Rank Sum Test + Log2fc
def filter_candidate_genes(df_filtered, meta_filtered):
    results = []
    for me_type, meta_g in meta_filtered.groupby("ME_Type"):
        group = TYPE_TO_GROUP.get(me_type)
        if group is None:
            continue
        group_members = GROUPS[group]
        samples_in = meta_g.index.tolist()
        df_in = df_filtered[samples_in]
        samples_out = meta_filtered.index[~meta_filtered["ME_Type"].isin(group_members)]
        df_out = df_filtered[samples_out]

        baseline_mean, baseline_std = df_out.mean(axis=1), df_out.std(axis=1)
        threshold_log = baseline_mean + 1.25 * baseline_std
        exceed = df_in.gt(threshold_log, axis=0)
        exceed_count = exceed.sum(axis=1)

        M = int(np.floor(len(samples_in) / 3))
        sample_to_patient = meta_g["patient_id"].to_dict()
        distinct_patients = exceed.apply(
            lambda row: len({sample_to_patient[s] for s in row[row].index}), axis=1
        )
        valid_genes = df_filtered.index[(exceed_count >= M) & (distinct_patients >= 4)]
        results.append(pd.DataFrame({"gene": valid_genes, "ME_Type": me_type}))
    return pd.concat(results, ignore_index=True)

def run_wilcoxon(df_filtered, meta_filtered, final_genes):
    records = []
    for _, row in final_genes.iterrows():
        gene, me_type = row["gene"], row["ME_Type"]
        group = TYPE_TO_GROUP.get(me_type)
        if group is None:
            continue
        group_members = GROUPS[group]
        samples_in = meta_filtered.index[meta_filtered["ME_Type"] == me_type]
        values_in = df_filtered.loc[gene, samples_in].dropna()
        samples_out = meta_filtered.index[~meta_filtered["ME_Type"].isin(group_members)]
        values_out = df_filtered.loc[gene, samples_out].dropna()
        stat, pval = mannwhitneyu(values_in, values_out, alternative="greater")
        records.append({
            "gene": gene, "ME_Type": me_type, "Group": group,
            "U_statistic": stat, "p_value": pval,
            "n_in": len(values_in), "n_out": len(values_out),
        })
    wilcoxon_df = pd.DataFrame(records)

    wilcoxon_df["FDR_within_ME"] = None
    for me_type, sub in wilcoxon_df.groupby("ME_Type"):
        fdr = multipletests(sub["p_value"], method="fdr_bh")[1]
        wilcoxon_df.loc[sub.index, "FDR_within_ME"] = fdr

    type_medians = df_filtered.T.groupby(meta_filtered["ME_Type"]).median().T

    def calc_log2fc(row):
        group = TYPE_TO_GROUP.get(row["ME_Type"])
        if group is None:
            return np.nan
        m_in = type_medians.loc[row["gene"], row["ME_Type"]]
        samples_out = meta_filtered.index[~meta_filtered["ME_Type"].isin(GROUPS[group])]
        m_out = df_filtered.loc[row["gene"], samples_out].median()
        return m_in - m_out

    wilcoxon_df["log2FC"] = wilcoxon_df.apply(calc_log2fc, axis=1)
    return wilcoxon_df.sort_values(by=["ME_Type", "FDR_within_ME"]).reset_index(drop=True)

def get_expanded_ms_genes(wilcoxon_df_sorted):
    genes = (
        wilcoxon_df_sorted.loc[
            (wilcoxon_df_sorted["FDR_within_ME"] < 0.05) &
            (wilcoxon_df_sorted["log2FC"] >= 1) &
            (wilcoxon_df_sorted["ME_Type"] != "EX"),
            "gene"
        ].dropna().str.split(";").explode().str.strip().unique()
    )
    return list(genes)

In [7]:
# NCI / DGI-db drug-target annotation
def base_name(drug_name):
    return drug_name.strip().upper().split(" ")[0]


def parse_nci_entry(entry):
    entry = entry.replace("\xa0", " ").strip()
    m = re.match(r"^(.*?)\s*\((.*?)\)$", entry)
    return (m.group(1).strip(), m.group(2).strip()) if m else (entry, entry)


def is_likely_regimen(name):
    letters_only = re.sub(r"[^A-Za-z]", "", name)
    return letters_only.isupper() and len(letters_only) > 0


def fetch_nci_drugs_for_cancer_type(url):
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    return [a.get_text(strip=True) for a in
            soup.select("main a[href*='/about-cancer/treatment/drugs/']") if a.get_text(strip=True)]


def get_nci_single_drug_set_for_types(cancer_type_urls):
    all_raw = []
    for _, url in cancer_type_urls.items():
        all_raw.extend(fetch_nci_drugs_for_cancer_type(url))
        time.sleep(0.5)
    parsed = [parse_nci_entry(e) for e in all_raw]
    single_drugs = set()
    for brand, generic in parsed:
        if brand != generic or not is_likely_regimen(generic):
            single_drugs.add(generic.strip().upper())
    return single_drugs


def _fetch_dgidb_nodes(gene_list, batch_size=100, pause=0.5):
    nodes = []
    genes = list(dict.fromkeys(str(g) for g in gene_list))
    query = """
    query GeneInfo($genes: [String!]) {
      genes(names: $genes) { nodes { name interactions {
        drug { name approved } interactionTypes { type } } } }
    }"""
    for i in range(0, len(genes), batch_size):
        batch = genes[i:i + batch_size]
        resp = requests.post(DGIDB_URL, json={"query": query, "variables": {"genes": batch}})
        resp.raise_for_status()
        data = resp.json()
        if "errors" in data:
            continue
        nodes.extend(data["data"]["genes"]["nodes"])
        time.sleep(pause)
    return nodes


def _fetch_withdrawn_drugs(drug_names, batch_size=100, pause=0.5):
    query = """
    query DrugInfo($names: [String!]) {
      drugs(names: $names) { nodes { name drugApprovalRatings { rating } } }
    }"""
    withdrawn = set()
    names = list(dict.fromkeys(str(d) for d in drug_names))
    for i in range(0, len(names), batch_size):
        batch = names[i:i + batch_size]
        resp = requests.post(DGIDB_URL, json={"query": query, "variables": {"names": batch}})
        resp.raise_for_status()
        data = resp.json()
        if "errors" in data:
            continue
        for node in data["data"]["drugs"]["nodes"]:
            ratings = [r["rating"] for r in node.get("drugApprovalRatings", [])]
            if "Withdrawn" in ratings:
                withdrawn.add(node["name"])
        time.sleep(pause)
    return withdrawn


def annotate_approved_targets(df, gene_col="gene"):
    genes = df[gene_col].unique().tolist()
    nodes = _fetch_dgidb_nodes(genes)
    all_approved_drugs = {
        i["drug"]["name"] for node in nodes for i in node.get("interactions", [])
        if i["drug"]["approved"] and i.get("interactionTypes")
    }
    withdrawn_drugs = _fetch_withdrawn_drugs(list(all_approved_drugs))

    approved_lookup, non_approved_lookup = {}, {}
    for node in nodes:
        hits, flag = [], False
        for interaction in node.get("interactions", []):
            types = interaction.get("interactionTypes", [])
            if not types:
                continue
            drug_name, is_approved = interaction["drug"]["name"], interaction["drug"]["approved"]
            if is_approved and drug_name not in withdrawn_drugs:
                hits.extend((drug_name, t["type"]) for t in types)
            elif not is_approved:
                flag = True
        approved_lookup[node["name"]] = hits
        non_approved_lookup[node["name"]] = flag

    df["approved_hits"] = df[gene_col].map(lambda g: approved_lookup.get(g, []))
    df["approved_drug_target"] = df["approved_hits"].map(lambda hits: len(hits) > 0)
    df["non_approved_drug_target"] = df[gene_col].map(lambda g: non_approved_lookup.get(g, False))
    return df


def fetch_dgidb_drug_targets(drug_names, batch_size=100, pause=0.5):
    query = """
    query DrugGenes($names: [String!]) {
      drugs(names: $names) { nodes { name interactions {
        gene { name } interactionTypes { type } } } }
    }"""
    targets = {}
    names = list(dict.fromkeys(str(d) for d in drug_names))
    for i in range(0, len(names), batch_size):
        batch = names[i:i + batch_size]
        resp = requests.post(DGIDB_URL, json={"query": query, "variables": {"names": batch}})
        resp.raise_for_status()
        data = resp.json()
        if "errors" in data:
            continue
        for node in data["data"]["drugs"]["nodes"]:
            gene_types = {}
            for ix in node.get("interactions", []):
                types = ix.get("interactionTypes", [])
                if not types or not ix.get("gene"):
                    continue
                gene_types.setdefault(ix["gene"]["name"], set()).update(t["type"] for t in types)
            targets[node["name"]] = gene_types
        time.sleep(pause)
    return targets

def annotate_wilcoxon_with_targets(wilcoxon_df_sorted):
    """Adds nci_approved_cancer_drug + investigational_pdac_target columns"""
    wilcoxon_df_sorted = annotate_approved_targets(wilcoxon_df_sorted)

    nci_single_drugs = get_nci_single_drug_set_for_types(NCI_CANCER_TYPE_URLS)
    nci_base_lookup = {}
    for name in nci_single_drugs:
        nci_base_lookup.setdefault(base_name(name), []).append(name)

    def get_nci_matched_drugs_with_salts(hits):
        matched = []
        for drug, itype in hits:
            drug_upper = drug.strip().upper()
            if drug_upper in nci_single_drugs or nci_base_lookup.get(base_name(drug_upper)):
                matched.append((drug, itype))
        return matched

    wilcoxon_df_sorted["nci_approved_hits"] = wilcoxon_df_sorted["approved_hits"].map(get_nci_matched_drugs_with_salts)
    wilcoxon_df_sorted["nci_approved_cancer_drug"] = wilcoxon_df_sorted["nci_approved_hits"].map(lambda h: len(h) > 0)

    wilcoxon_df_sorted["investigational_pdac_target"] = wilcoxon_df_sorted["gene"].map(INVESTIGATIONAL_PDAC_TARGETS.get)
    wilcoxon_df_sorted["is_investigational_pdac_target"] = wilcoxon_df_sorted["investigational_pdac_target"].notna()
    return wilcoxon_df_sorted, nci_single_drugs

def compute_extended_gene_list(df_filtered, meta_filtered, df_unimputed, mean_expr_ME,
                                wilcoxon_annotated, nci_single_drugs):
    nci_drug_targets = fetch_dgidb_drug_targets(nci_single_drugs)
    records = [{"drug": d, "gene": g} for d, gt in nci_drug_targets.items() for g in gt]
    df_nci_drug_targets = pd.DataFrame(records)

    all_genes_in_data = {g.strip() for ms in df_filtered.index for g in str(ms).split(";")}
    df_nci_drug_targets["in_df_filtered"] = df_nci_drug_targets["gene"].isin(all_genes_in_data)
    nci_genes_found = df_nci_drug_targets.loc[df_nci_drug_targets["in_df_filtered"], "gene"].unique()

    gene_to_msgene = {}
    for ms_gene in df_filtered.index:
        for g in str(ms_gene).split(";"):
            gene_to_msgene.setdefault(g.strip(), []).append(ms_gene)

    nonnan_frac = df_unimputed.notna().T.groupby(meta_filtered["ME_Type"]).mean().T
    me_types_full = [c for c in ME_ORDER if c in mean_expr_ME.columns]

    all_target_genes = sorted(set(nci_genes_found) | set(INVESTIGATIONAL_PDAC_TARGETS.keys()))
    records = []
    for gene in all_target_genes:
        for ms_gene in gene_to_msgene.get(gene, []):
            if ms_gene not in nonnan_frac.index:
                continue
            frac_row = nonnan_frac.loc[[ms_gene], ME_TYPES_CHECK].max(axis=0)
            if (frac_row >= 0.25).all():
                expr_row = mean_expr_ME.loc[[ms_gene], me_types_full].mean(axis=0)
                records.append({"gene": gene, "MS_gene": ms_gene, "max_ME_Type": expr_row.idxmax()})

    df_completeness = pd.DataFrame(records)
    return sorted(df_completeness["gene"].unique())

def build_subset_for_gene_list(genes, mean_expr_ME, tau_ms):
    gene_to_msgene = {}
    for ms_gene in mean_expr_ME.index:
        for g in str(ms_gene).split(";"):
            gene_to_msgene.setdefault(g.strip(), []).append(ms_gene)

    records = []
    missing = []
    for gene in genes:
        ms_genes = gene_to_msgene.get(gene, [])
        if not ms_genes:
            missing.append(gene)
            continue
        for ms_gene in ms_genes:
            records.append({
                "gene": gene,
                "MS_gene": ms_gene,
                "top_ME_Type": mean_expr_ME.loc[ms_gene].idxmax(),
                "tau_MS": tau_ms.get(ms_gene, np.nan),
            })
    return pd.DataFrame(records)

In [8]:
# scRNA-seq data for plotting
def rebuild_scrna(gene_set, out_path):
    """Subsets each raw compartment object to `gene_set`, writes intermediate
    files, concatenates, cleans up, and deletes intermediates as it goes so
    only one full compartment object is ever in memory at once."""
    compartment_paths = {}
    for key, (path, label_col) in RAW_SC_PATHS.items():
        a = sc.read_h5ad(path)
        a.var_names_make_unique()
        if key == "ep":
            a = a[a.obs["cell_type"].notna()].copy()
        else:
            a.obs["cell_type"] = a.obs[label_col]
        a = a[:, a.var_names.isin(gene_set)].copy()
        tmp_path = OUT_DIR + f"{key}_extended_genes.h5ad"
        a.write(tmp_path)
        compartment_paths[key] = tmp_path
        del a
        gc.collect()

    pieces = {k: sc.read_h5ad(p) for k, p in compartment_paths.items()}
    adata = ad.concat(pieces, join="outer", label="compartment")
    del pieces
    gc.collect()

    mask = (~adata.obs["cell_type"].isna()) & (~adata.obs["cell_type"].isin(["undefined", "nan"]))
    adata = adata[mask].copy()

    cols_to_drop = ['singler.pruned.label', 'CAF_cluster_labels', 'Endothelial_cluster_labels',
                    'cell type', 'test', 'Cell Type', 'leiden']
    adata.obs = adata.obs.drop(columns=[c for c in cols_to_drop if c in adata.obs.columns])

    adata.obs["cell_type"] = pd.Categorical(adata.obs["cell_type"], categories=CELL_TYPE_ORDER, ordered=True)
    adata = adata[adata.obs.sort_values("cell_type").index].copy()

    adata.write(out_path)
    return adata

In [9]:
# Tau cell-type specificity filter + helper functions for heatmaps, Figure 4

def compute_tau(adata, group_key="cell_type", use_raw=False, agg="mean"):
    X = adata.raw.X if use_raw and adata.raw is not None else adata.X
    if not isinstance(X, np.ndarray):
        X = X.toarray()
    groups = adata.obs[group_key].astype(str)
    unique_groups = groups.unique()
    expr_by_group = []
    for g in unique_groups:
        idx = np.where(groups == g)[0]
        expr_by_group.append(X[idx].mean(axis=0) if agg == "mean" else np.median(X[idx], axis=0))
    expr_by_group = np.vstack(expr_by_group)
    max_expr = expr_by_group.max(axis=0)
    max_expr[max_expr == 0] = 1e-12
    xhat = expr_by_group / max_expr
    tau = (1 - xhat).sum(axis=0) / (len(unique_groups) - 1)
    return pd.Series(tau, index=adata.var_names, name="tau")


def compute_celltype_means(adata):
    return adata.to_df().join(adata.obs['cell_type']).groupby('cell_type').mean().T


def compute_tau_ms(mean_expr_ME):
    expr = mean_expr_ME.copy()
    expr = expr.sub(expr.min(axis=1), axis=0)

    def tau_for_row(v):
        m = v.max()
        if m <= 0:
            return 0.0
        xhat = v / m
        return (1 - xhat).sum() / (len(v) - 1)

    tau_ms = expr.apply(tau_for_row, axis=1)
    tau_ms.name = "Tau_MS"
    return tau_ms

def build_summary_table(wilcoxon_annotated, mean_expr_ME, tau, adata):
    ms_gene_entries = (
        wilcoxon_annotated.loc[
            (wilcoxon_annotated["FDR_within_ME"] < 0.05) &
            (wilcoxon_annotated["log2FC"] >= 1) &
            (wilcoxon_annotated["ME_Type"] != "EX"),
            "gene"
        ].dropna().unique()
    )
    ms_to_sc = {ms: [g.strip() for g in ms.split(";")] for ms in ms_gene_entries}

    records = []
    for ms_gene, sc_genes in ms_to_sc.items():
        if ms_gene not in mean_expr_ME.index:
            continue
        sig_me_types = wilcoxon_annotated.loc[
            (wilcoxon_annotated["gene"] == ms_gene) &
            (wilcoxon_annotated["FDR_within_ME"] < 0.05) &
            (wilcoxon_annotated["log2FC"] >= 1) &
            (wilcoxon_annotated["ME_Type"] != "EX"),
            "ME_Type"
        ].unique()
        expr = mean_expr_ME.loc[ms_gene]
        if len(sig_me_types) == 0:
            top_ME = np.nan
        else:
            sig_expr = expr[sig_me_types]
            threshold = 0.9 * expr.max()
            sig_expr = sig_expr[sig_expr >= threshold]
            top_ME = sig_expr.idxmax() if len(sig_expr) else np.nan
        for g in sc_genes:
            records.append({
                "MS_gene": ms_gene, "gene": g,
                "tau": tau[g] if g in tau.index else np.nan,
                "top_ME_Type": top_ME,
            })
    return pd.DataFrame(records)


def filter_broad_group_specificity(subset, celltype_means, threshold=0.75):
    """keep only genes whose expression inside their broad compartment
    (Malignant/Immune/Stroma) is within `threshold` of the max outside it."""
    subset = subset.copy()
    subset["broad_group"] = subset["top_ME_Type"].map(BROAD_ME_TO_GROUP)

    malignant_max = celltype_means[[c for c in MALIGNANT_CT if c in celltype_means.columns]].max(axis=1)
    immune_max = celltype_means[[c for c in IMMUNE_CT if c in celltype_means.columns]].max(axis=1)
    stroma_max = celltype_means[[c for c in STROMA_CT if c in celltype_means.columns]].max(axis=1)
    broad_means = pd.DataFrame({"Malignant": malignant_max, "Immune": immune_max, "Stroma": stroma_max})

    inside_max, outside_max = [], []
    for gene, row in broad_means.iterrows():
        grp = subset.loc[subset["gene"] == gene, "broad_group"]
        if grp.empty:
            inside_max.append(None); outside_max.append(None)
            continue
        grp = grp.iloc[0]
        if grp in ["Malignant", "Immune", "Stroma"]:
            inside_max.append(row[grp])
            outside_max.append(row.drop(labels=[grp]).max())
        elif grp in ["IS", "CCF"]:
            inside_max.append(max(row["Immune"], row["Stroma"]))
            outside_max.append(row["Malignant"])
        else:
            inside_max.append(None); outside_max.append(None)

    comparison_df = pd.DataFrame({"inside_max": inside_max, "outside_max": outside_max}, index=broad_means.index)
    subset = subset.merge(comparison_df, left_on="gene", right_index=True, how="left")
    return subset[subset["inside_max"] > threshold * subset["outside_max"]].copy()


def filter_and_merge_subset(summary_table, mean_expr_ME, tau_ms, wilcoxon_annotated, tau_cutoff=0.6):
    subset = summary_table[(summary_table.tau > tau_cutoff)].dropna(subset=["top_ME_Type"]).copy()

    ex_dominant = mean_expr_ME.index[mean_expr_ME["EX"] > 0.95 * mean_expr_ME.max(axis=1)]
    subset = subset[~subset["MS_gene"].isin(ex_dominant)]

    subset["tau_MS"] = subset["MS_gene"].map(tau_ms)
    subset = subset.sort_values(["top_ME_Type", "tau_MS"], ascending=[True, False])

    wilcoxon_renamed = wilcoxon_annotated.rename(columns={"gene": "MS_gene"})
    merged = subset.merge(wilcoxon_renamed, left_on=["MS_gene", "top_ME_Type"],
                           right_on=["MS_gene", "ME_Type"], how="left")
    return subset, merged

In [10]:
# Heatmaps (Figure 4)
from matplotlib.patches import Rectangle

def _broad_map_and_colors():
    broad_map = {}
    for ct in ["arteries", "capillaries",# "CCL21+ PDPN+ lymphatics", 
               "CD36+ capillaries", "ESM1+ Tip cells", "veins"]:
        broad_map[ct] = "Endothelial"
    for ct in ["B lymphocytes", "CD4-positive  alpha-beta T cell", "CD8-positive  alpha-beta T cell", "T regs",
               "NK cells", "Macrophages", "mDC", "pDC", "Monocytes", "MDSCs", "Mast cells", "RBCs"]:
        broad_map[ct] = "Hematopoetic"
    for ct in ["CCL21+ Reticular CAFs", "CD105+ CAFs", "CLU+ Vascular smooth muscle cells",
               #"CXCL13+ CLU+ CD105+ CAFs", 
               "POSTN+ CD105+ CAFs", "iCAF 1", "iCAF 2", "myCAF 1", "myCAF 2",
               "Vascular smooth muscle cells", "Pericytes", "Schwann cells"]:
        broad_map[ct] = "CAFS"
    for ct in ["edTFnull_Classical", "edTFlow_Classical", "edTFhigh_Classical", "Basal",
               "S100A4+_Classical-like", "Hybrid", "tumor-associated normal"]:
        broad_map[ct] = "pancreatic ductal cell"
    broad_color_dict = {"Endothelial": "#FFC5CD", "Hematopoetic": "#8754A2",
                         "CAFS": "#A48870", "pancreatic ductal cell": "#8B0000"}
    return broad_map, broad_color_dict


def _gene_to_pdac_map(df_PDACGenesets):
    activated_sets = {"Moffitt.Activated.25"}
    classical_sets = {"Collisson.Classical", "Chan_Seng_Yue.ClassicalA", "Chan_Seng_Yue.ClassicalB", "Moffitt.Tumor_classical"}
    basal_sets = {"Chan_Seng_Yue.BasalA", "Chan_Seng_Yue.BasalB", "Moffitt.Tumor_basal", "Collisson.QM"}
    gene_to_pdac = {}
    for _, row in df_PDACGenesets.iterrows():
        geneset, g = row["geneset"], row["gene"]
        if geneset in activated_sets:
            gene_to_pdac[g] = "Activated"
        elif geneset in classical_sets:
            gene_to_pdac[g] = "Classical"
        elif geneset in basal_sets:
            gene_to_pdac[g] = "Basal"
    return gene_to_pdac


def build_heatmap_block(subset, mean_expr_ME, celltype_means, df_geneanno, label_all_genes=False, zscore_ms=False, wilcoxon_annotated=None):
    if wilcoxon_annotated is not None:
        # broad: any DGIdb-approved drug target -- controls which genes get a
        # left-side label at all. Keyed by MS_gene (the raw protein-group
        # string, e.g. "AKT1;AKT2;AKT3")
        approved_lookup = wilcoxon_annotated.groupby("gene")["approved_drug_target"].any()
        # narrow: NCI-restricted approval -- controls which labeled genes get bolded
        nci_approved_lookup = wilcoxon_annotated.groupby("gene")["nci_approved_cancer_drug"].any()
    else:
        approved_lookup = None
        nci_approved_lookup = None

    investigational_genes = set(INVESTIGATIONAL_PDAC_TARGETS.keys())

    all_me_blocks, all_ct_blocks, all_labels, all_target_labels, all_target_bold = [], [], [], [], []

    for me in ME_ORDER:
        module_df = subset.loc[subset["top_ME_Type"] == me]
        if module_df.empty:
            continue
        me_rows = module_df.sort_values("tau_MS", ascending=False)
        valid = me_rows["gene"].isin(celltype_means.index)
        me_rows = me_rows[valid]
        if me_rows.empty:
            continue

        df_me = mean_expr_ME.loc[me_rows["MS_gene"].values]
        df_me.index = me_rows["gene"].values
        df_ct = celltype_means.loc[me_rows["gene"].values]
        df_ct.index = me_rows["gene"].values

        df_me = df_me[[c for c in ME_ORDER if c in df_me.columns]]
        df_ct = df_ct[[c for c in CELL_TYPE_ORDER if c in df_ct.columns]]

        if zscore_ms:
            # row-wise z-score: (x - mean) / std, for the mass-spec (ME_Type) panel only
            mu = df_me.mean(axis=1)
            sigma = df_me.std(axis=1)
            df_me_norm = df_me.sub(mu, axis=0).div(sigma.replace(0, np.nan), axis=0)
            df_me_norm = df_me_norm.fillna(0)
        else:
            df_me_norm = (df_me - df_me.min(axis=1).values[:, None]) / (df_me.max(axis=1) - df_me.min(axis=1)).values[:, None]
            df_me_norm = df_me_norm.fillna(0)
        df_ct_norm = (df_ct - df_ct.min(axis=1).values[:, None]) / (df_ct.max(axis=1) - df_ct.min(axis=1)).values[:, None]
        df_ct_norm = df_ct_norm.fillna(0)

        dominant_ct = df_ct_norm.idxmax(axis=1)
        gene_anno_rows = df_geneanno.reindex(df_ct_norm.index)
        categories = gene_anno_rows["Annotation"]

        # look up approval by MS_gene (not the split single gene symbol) --
        # a gene's MS_gene entry can be compound (e.g. "AKT1;AKT2;AKT3"), and
        # reindexing by the plain symbol silently misses those rows
        ms_gene_for_row = pd.Series(me_rows["MS_gene"].values, index=me_rows["gene"].values)

        is_investigational = gene_anno_rows.index.to_series().isin(investigational_genes)
        if approved_lookup is not None:
            is_approved = ms_gene_for_row.reindex(gene_anno_rows.index).map(
                lambda ms: bool(approved_lookup.get(ms, False))
            )
            is_nci_approved = ms_gene_for_row.reindex(gene_anno_rows.index).map(
                lambda ms: bool(nci_approved_lookup.get(ms, False))
            )
        else:
            is_approved = pd.Series(False, index=gene_anno_rows.index)
            is_nci_approved = pd.Series(False, index=gene_anno_rows.index)
        bold_flag = is_nci_approved | is_investigational

        if approved_lookup is not None:
            target_flag = is_approved.copy()
        else:
            target_flag = bold_flag

        if label_all_genes:
            target_flag = pd.Series(True, index=gene_anno_rows.index)

        annotated = categories.dropna()
        unannotated = categories[categories.isna()]
        ordered_genes = []
        for cat in annotated.unique():
            genes_in_cat = annotated[annotated == cat].index
            ordered_genes.extend(dominant_ct.loc[genes_in_cat].sort_values().index)

        # Within the remaining (non-annotated) genes, cluster the labelled
        # therapeutic-target genes together first, then everything else 
        # both sub-groups still sorted by dominant cell type.
        if len(unannotated) > 0:
            is_target = target_flag.reindex(unannotated.index, fill_value=False).astype(bool)
            unannotated_target = unannotated.index[is_target]
            unannotated_other = unannotated.index[~is_target]
            if len(unannotated_target) > 0:
                ordered_genes.extend(dominant_ct.loc[unannotated_target].sort_values().index)
            if len(unannotated_other) > 0:
                ordered_genes.extend(dominant_ct.loc[unannotated_other].sort_values().index)

        df_me_norm = df_me_norm.loc[ordered_genes]
        df_ct_norm = df_ct_norm.loc[ordered_genes]
        labels = [g if g in annotated.index else "" for g in ordered_genes]
        target_labels = [g if target_flag.get(g, False) else "" for g in ordered_genes]
        target_bold = [bool(bold_flag.get(g, False)) for g in ordered_genes]

        all_me_blocks.append(df_me_norm)
        all_ct_blocks.append(df_ct_norm)
        all_labels.extend(labels)
        all_target_labels.extend(target_labels)
        all_target_bold.extend(target_bold)

    big_me = pd.concat(all_me_blocks, axis=0)
    big_ct = pd.concat(all_ct_blocks, axis=0)
    return big_me, big_ct, all_labels, all_target_labels, all_target_bold


def _draw_block_heatmaps(big_me, big_ct, all_labels, all_target_labels, ax_me, ax_ct,
                          cmap_me, vmin_me, vmax_me, cmap_ct, vmin_ct, vmax_ct,
                          show_right_labels=True, show_left_labels=True,
                          show_x_labels_me=True, show_x_labels_ct=True,
                          all_target_bold=None):
    sns.heatmap(big_me, cmap=cmap_me, vmin=vmin_me, vmax=vmax_me, linewidths=0, ax=ax_me, yticklabels=big_me.index, cbar=False)
    sns.heatmap(big_ct, cmap=cmap_ct, vmin=vmin_ct, vmax=vmax_ct, linewidths=0, ax=ax_ct, yticklabels=big_ct.index, cbar=False)
    ax_ct.set_yticklabels(all_labels if show_right_labels else [], fontsize=4, rotation=0)
    ax_ct.yaxis.tick_right(); ax_ct.tick_params(axis='y', length=0)
    ax_me.set_yticklabels(all_target_labels if show_left_labels else [], fontsize=4, rotation=0)
    ax_me.yaxis.tick_left(); ax_me.tick_params(axis='y', length=0)
    if show_left_labels and all_target_bold is not None:
        for text_obj, is_bold in zip(ax_me.get_yticklabels(), all_target_bold):
            if is_bold and text_obj.get_text():
                text_obj.set_fontweight('bold')
    # seaborn auto-labels the x-axis with the DataFrame's columns.name
    # ("ME_Type" / "cell_type") \u2014 that title is redundant with the column
    # codes shown as tick labels, so always clear it.
    ax_me.set_xlabel(''); ax_ct.set_xlabel('')
    if not show_x_labels_me:
        ax_me.set_xticklabels([]); ax_me.tick_params(axis='x', length=0)
    if not show_x_labels_ct:
        ax_ct.set_xticklabels([]); ax_ct.tick_params(axis='x', length=0)


def _reorder_block_by_signature(big_me, big_ct, labels, target_labels, target_bold,
                                  gene_to_signature_info, signature_order):
    order_rank = {name: i for i, name in enumerate(signature_order)}
    multiple_rank = len(signature_order)       
    unassigned_rank = len(signature_order) + 1  

    def _group_rank(gene):
        entry = gene_to_signature_info.get(gene)
        if not entry:
            return unassigned_rank
        names = list(dict.fromkeys(entry["signatures"]))
        if len(names) > 1:
            return multiple_rank
        return order_rank.get(names[0], unassigned_rank)

    genes = list(big_me.index)
    ranked = sorted(range(len(genes)), key=lambda i: (_group_rank(genes[i]), i))  # stable within group

    new_index = [genes[i] for i in ranked]
    big_me = big_me.loc[new_index]
    big_ct = big_ct.loc[new_index]
    labels = [labels[i] for i in ranked]
    target_labels = [target_labels[i] for i in ranked]
    target_bold = [target_bold[i] for i in ranked]
    return big_me, big_ct, labels, target_labels, target_bold


def _draw_pdac_strip(ax_anno, gene_index, gene_to_pdac):
    pdac_color_dict = {"Activated": "#3C5934", "Classical": "#7D8E9E", "Basal": "#A0BEC0", None: "#FFFFFF"}
    pdac_colors = [pdac_color_dict.get(gene_to_pdac.get(g), "#FFFFFF") for g in gene_index]
    pdac_rgb = np.array([to_rgb(c) for c in pdac_colors])
    y_pdac = np.arange(len(pdac_rgb) + 1)
    x_pdac = np.array([0, 1])
    ax_anno.pcolormesh(x_pdac, y_pdac, pdac_rgb.reshape(len(pdac_rgb), 1, 3), shading='flat')
    ax_anno.set_xlim(0, 1); ax_anno.set_ylim(len(pdac_rgb), 0); ax_anno.set_xticks([]); ax_anno.set_yticks([])

_CB_SAFE_BASE = [
    "#000000", "#E69F00", "#56B4E9", "#009E73",
    "#F0E442", "#0072B2", "#D55E00", "#CC79A7",
]


def _signature_color_dict(signature_names):
    ordered_names = sorted(signature_names)
    n = len(ordered_names)
    if n == 0:
        return {}
    palette = []
    variant = 0
    while len(palette) < n:
        base_hex = _CB_SAFE_BASE[len(palette) % len(_CB_SAFE_BASE)]
        rgb = to_rgb(base_hex)
        if variant == 0:
            shade = rgb
        elif variant % 2 == 1:
            factor = 0.6 ** ((variant + 1) // 2)  # darken
            shade = tuple(c * factor for c in rgb)
        else:
            factor = 0.35 * (variant // 2)  # lighten toward white
            shade = tuple(c + (1 - c) * factor for c in rgb)
        palette.append(shade)
        if len(palette) % len(_CB_SAFE_BASE) == 0:
            variant += 1
    return {name: palette[i] for i, name in enumerate(ordered_names)}

_DIRECTION_COLORS = {"Up": "#D62828", "Down": "#1D3557", "Mixed": "#6A4C93", None: "#FFFFFF"}
_MULTIPLE_SIGNATURE_COLOR = "#999999"
_MULTIPLE_SIGNATURE_LABEL = "Multiple signatures"


def _draw_signature_tracks(ax_sig, ax_dir, gene_index, target_labels, target_bold,
                            gene_to_signature_info, label_fontsize=4):
    per_gene_sigs, dir_values = [], []
    for g in gene_index:
        entry = gene_to_signature_info.get(g)
        if entry:
            names = list(dict.fromkeys(entry["signatures"]))  # de-dup, keep order
            per_gene_sigs.append(names)
            dir_values.append(entry["direction"])
        else:
            per_gene_sigs.append([])
            dir_values.append(None)

    # direction strip (unchanged logic, new color scheme)
    dir_colors = [_DIRECTION_COLORS.get(d, "#FFFFFF") for d in dir_values]
    dir_rgb = np.array([to_rgb(c) for c in dir_colors])
    y_dir = np.arange(len(dir_rgb) + 1)
    x_dir = np.array([0, 1])
    ax_dir.pcolormesh(x_dir, y_dir, dir_rgb.reshape(len(dir_rgb), 1, 3), shading='flat')
    ax_dir.set_xlim(0, 1); ax_dir.set_ylim(len(dir_rgb), 0); ax_dir.set_xticks([]); ax_dir.set_yticks([])

    # signature track: one color per gene (single signature / "Multiple" / blank)
    single_sig_names = {names[0] for names in per_gene_sigs if len(names) == 1}
    sig_color_dict = _signature_color_dict(single_sig_names)
    n_multiple = sum(1 for names in per_gene_sigs if len(names) > 1)

    def _color_for(names):
        if len(names) == 0:
            return "#FFFFFF"
        if len(names) > 1:
            return _MULTIPLE_SIGNATURE_COLOR
        return sig_color_dict[names[0]]

    sig_colors = [_color_for(names) for names in per_gene_sigs]
    sig_rgb = np.array([to_rgb(c) for c in sig_colors])
    y_sig = np.arange(len(sig_rgb) + 1)
    x_sig = np.array([0, 1])
    ax_sig.pcolormesh(x_sig, y_sig, sig_rgb.reshape(len(sig_rgb), 1, 3), shading='flat')
    ax_sig.set_xlim(0, 1); ax_sig.set_ylim(len(sig_rgb), 0); ax_sig.set_xticks([])

    # gene-name labels live here (leftmost axis = only place with free margin)
    ax_sig.set_yticks(np.arange(len(target_labels)) + 0.5)
    ax_sig.set_yticklabels(target_labels, fontsize=label_fontsize, rotation=0)
    ax_sig.yaxis.tick_left(); ax_sig.tick_params(axis='y', length=0)
    if target_bold is not None:
        for text_obj, is_bold in zip(ax_sig.get_yticklabels(), target_bold):
            if is_bold and text_obj.get_text():
                text_obj.set_fontweight('bold')

    return sig_color_dict, n_multiple


def plot_paired_heatmap(subset, mean_expr_ME, celltype_means, df_geneanno,
                         fine_color_dict, df_PDACGenesets, out_path,
                         celltype_order=None, me_order=None,
                         fig_height=30, row_height=None, cmap_color="#4527A0",
                         label_all_genes=False, show_right_labels=True,
                         show_left_labels=True, show_legend=True,
                         show_x_labels_me=True, show_x_labels_ct=True,
                         zscore_mass_spec=False, zscore_cmap="RdBu_r", zscore_vmax=2.0,
                         wilcoxon_annotated=None):
    cmap = sns.light_palette(cmap_color, as_cmap=True)
    broad_map, broad_color_dict = _broad_map_and_colors()
    gene_to_pdac = _gene_to_pdac_map(df_PDACGenesets)

    big_me, big_ct, all_labels, all_target_labels, all_target_bold = build_heatmap_block(
        subset, mean_expr_ME, celltype_means, df_geneanno, label_all_genes=label_all_genes,
        zscore_ms=zscore_mass_spec, wilcoxon_annotated=wilcoxon_annotated,
    )
    if zscore_mass_spec:
        cmap_me, vmin_me, vmax_me = zscore_cmap, -zscore_vmax, zscore_vmax
    else:
        cmap_me, vmin_me, vmax_me = cmap, 0, 1
    cmap_ct, vmin_ct, vmax_ct = cmap, 0, 1

    missing_colors = [c for c in big_ct.columns if c not in fine_color_dict]
    if missing_colors:
        print(f"[plot_paired_heatmap] no color assigned for: {missing_colors} \u2014 using gray (#CCCCCC)")
    fine_colors = [fine_color_dict.get(c, "#CCCCCC") for c in big_ct.columns]
    missing_broad = [c for c in big_ct.columns if c not in broad_map]
    if missing_broad:
        print(f"[plot_paired_heatmap] no broad-group assigned for: {missing_broad} \u2014 using gray (#CCCCCC)")
    broad_colors = [broad_color_dict.get(broad_map.get(c), "#CCCCCC") for c in big_ct.columns]
    fine_rgb = np.array([to_rgb(c) for c in fine_colors])
    broad_rgb = np.array([to_rgb(c) for c in broad_colors])

    n_rows = len(big_me)
    height = max(6, n_rows * row_height) if row_height is not None else fig_height
    fig = plt.figure(figsize=(22, height))
    gs = fig.add_gridspec(3, 3, height_ratios=[0.2, 0.2, 20], width_ratios=[1, 0.03, 1],
                           hspace=0.005, wspace=0.05)

    ax_broad = fig.add_subplot(gs[0, 2])
    ax_fine = fig.add_subplot(gs[1, 2])
    ax_me = fig.add_subplot(gs[2, 0])
    ax_anno = fig.add_subplot(gs[2, 1])
    ax_ct = fig.add_subplot(gs[2, 2])

    x_top = np.arange(len(fine_rgb) + 1)
    y_top = np.array([0, 1])
    ax_broad.pcolormesh(x_top, y_top, broad_rgb.reshape(1, len(broad_rgb), 3), shading='flat')
    ax_broad.set_xlim(0, len(broad_rgb)); ax_broad.set_ylim(0, 1); ax_broad.set_xticks([]); ax_broad.set_yticks([])
    ax_fine.pcolormesh(x_top, y_top, fine_rgb.reshape(1, len(fine_rgb), 3), shading='flat')
    ax_fine.set_xlim(0, len(fine_rgb)); ax_fine.set_ylim(0, 1); ax_fine.set_xticks([]); ax_fine.set_yticks([])

    _draw_block_heatmaps(big_me, big_ct, all_labels, all_target_labels, ax_me, ax_ct,
                          cmap_me, vmin_me, vmax_me, cmap_ct, vmin_ct, vmax_ct,
                          show_right_labels=show_right_labels, show_left_labels=show_left_labels,
                          show_x_labels_me=show_x_labels_me, show_x_labels_ct=show_x_labels_ct,
                          all_target_bold=all_target_bold)

    _draw_pdac_strip(ax_anno, big_me.index, gene_to_pdac)
    ax_anno.set_ylim(ax_me.get_ylim())

    for ax in [ax_broad, ax_fine, ax_anno, ax_me, ax_ct]:
        for spine in ax.spines.values():
            spine.set_linewidth(0.8); spine.set_edgecolor("black"); spine.set_visible(True)

    ax_cbar = ax_ct.inset_axes([1.05, 0.0, 0.025, 0.2], transform=ax_ct.transAxes)
    sm = ScalarMappable(cmap=cmap_ct, norm=plt.Normalize(vmin=vmin_ct, vmax=vmax_ct))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=ax_cbar)
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label("Normalized Expression", fontsize=12)

    if zscore_mass_spec:
        ax_cbar_me = ax_me.inset_axes([-0.22, 0.0, 0.025, 0.2], transform=ax_me.transAxes)
        sm_me = ScalarMappable(cmap=cmap_me, norm=plt.Normalize(vmin=vmin_me, vmax=vmax_me))
        sm_me.set_array([])
        cbar_me = fig.colorbar(sm_me, cax=ax_cbar_me)
        cbar_me.ax.tick_params(labelsize=10)
        cbar_me.ax.yaxis.set_label_position('left')
        cbar_me.ax.yaxis.set_ticks_position('right')
        cbar_me.set_label("Mass spec (z-score)", fontsize=12, labelpad=8)

    if show_legend:
        pdac_color_dict = {"Activated": "#3C5934", "Classical": "#7D8E9E", "Basal": "#A0BEC0", None: "#FFFFFF"}
        broad_handles = [Patch(facecolor=broad_color_dict[k], edgecolor="black", label=k) for k in broad_color_dict]
        fine_handles = [Patch(facecolor=fine_color_dict.get(c, "#CCCCCC"), edgecolor="black", label=c) for c in big_ct.columns]
        pdac_handles = [Patch(facecolor=pdac_color_dict[k], edgecolor="black", label=str(k)) for k in ["Activated", "Classical", "Basal", None]]
        legend_y = 1 + (0.6 / height)
        leg1 = fig.legend(handles=broad_handles, title="Broad Cell Types", loc="upper left", bbox_to_anchor=(0.01, legend_y), frameon=True)
        leg2 = fig.legend(handles=fine_handles, title="Fine Cell Types", loc="upper center", bbox_to_anchor=(0.55, legend_y), ncol=4, frameon=True)
        leg3 = fig.legend(handles=pdac_handles, title="PDAC Class", loc="upper right", bbox_to_anchor=(0.99, legend_y), frameon=True)
        fig.add_artist(leg1); fig.add_artist(leg2); fig.add_artist(leg3)

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return big_me, big_ct


def plot_stacked_heatmaps(blocks, mean_expr_ME, celltype_means, df_geneanno, fine_color_dict,
                           df_PDACGenesets, out_path, cmap_color="#645EA4", #"#4527A0",
                           default_row_height=None, min_block_height=2.0, gap_height=0.2,
                           zscore_mass_spec=False, zscore_cmap="RdBu_r", zscore_vmax=2.0,
                           wilcoxon_annotated=None, gene_to_signature_info=None, signature_order=None):
    cmap = sns.light_palette(cmap_color, as_cmap=True)
    broad_map, broad_color_dict = _broad_map_and_colors()
    gene_to_pdac = _gene_to_pdac_map(df_PDACGenesets)

    built = []
    for b in blocks:
        big_me, big_ct, labels, target_labels, target_bold = build_heatmap_block(
            b["subset"], mean_expr_ME, celltype_means, df_geneanno,
            label_all_genes=b.get("label_all_genes", False),
            zscore_ms=zscore_mass_spec, wilcoxon_annotated=wilcoxon_annotated,
        )
        if b.get("no_bold", False):
            target_bold = [False] * len(target_bold)
        if b.get("bold_all_genes", False):
            target_bold = [True] * len(target_bold)
        if b.get("order_by_signature", False):
            if gene_to_signature_info is None or signature_order is None:
                raise ValueError(
                    "block has order_by_signature=True but gene_to_signature_info "
                    "and/or signature_order was not passed to plot_stacked_heatmaps"
                )
            big_me, big_ct, labels, target_labels, target_bold = _reorder_block_by_signature(
                big_me, big_ct, labels, target_labels, target_bold,
                gene_to_signature_info, signature_order,
            )
        built.append((big_me, big_ct, labels, target_labels, target_bold, b))

    if zscore_mass_spec:
        cmap_me, vmin_me, vmax_me = zscore_cmap, -zscore_vmax, zscore_vmax
    else:
        cmap_me, vmin_me, vmax_me = cmap, 0, 1
    cmap_ct, vmin_ct, vmax_ct = cmap, 0, 1

    total_rows = sum(len(bm) for bm, *_ in built) or 1
    fallback_row_height = default_row_height if default_row_height is not None else 30 / total_rows

    block_heights = []
    for big_me, *_rest, b in built:
        if "height" in b:
            block_heights.append(b["height"])
        else:
            rh = b.get("row_height", fallback_row_height)
            block_heights.append(max(min_block_height, len(big_me) * rh))

    strip_height = 0.4
    gap = gap_height
    total_height = strip_height + sum(block_heights) + gap * (len(built) - 1)

    fig = plt.figure(figsize=(22, total_height))

    height_ratios = [0.2, 0.2]
    for i, bh in enumerate(block_heights):
        height_ratios.append(bh)
        if i < len(block_heights) - 1:
            height_ratios.append(gap)
    n_rows_gs = len(height_ratios)
    # 5 columns: [signature name, up/down strip, mass-spec panel, PDAC-class
    # strip, single-cell panel]. 
    gs = fig.add_gridspec(n_rows_gs, 5, height_ratios=height_ratios,
                           width_ratios=[0.045, 0.045, 1, 0.03, 1],
                           hspace=0.02, wspace=0.05)

    all_cols = celltype_means.columns
    fine_colors = [fine_color_dict.get(c, "#CCCCCC") for c in all_cols]
    missing_colors = [c for c in all_cols if c not in fine_color_dict]
    if missing_colors:
        print(f"[plot_stacked_heatmaps] no color assigned for: {missing_colors} — using gray (#CCCCCC)")
    broad_colors = [broad_color_dict.get(broad_map.get(c), "#CCCCCC") for c in all_cols]
    missing_broad = [c for c in all_cols if c not in broad_map]
    if missing_broad:
        print(f"[plot_stacked_heatmaps] no broad-group assigned for: {missing_broad} — using gray (#CCCCCC)")
    fine_rgb = np.array([to_rgb(c) for c in fine_colors])
    broad_rgb = np.array([to_rgb(c) for c in broad_colors])

    ax_broad = fig.add_subplot(gs[0, 4])
    ax_fine = fig.add_subplot(gs[1, 4])
    x_top = np.arange(len(fine_rgb) + 1)
    y_top = np.array([0, 1])
    ax_broad.pcolormesh(x_top, y_top, broad_rgb.reshape(1, len(broad_rgb), 3), shading='flat')
    ax_broad.set_xlim(0, len(broad_rgb)); ax_broad.set_ylim(0, 1); ax_broad.set_xticks([]); ax_broad.set_yticks([])
    ax_fine.pcolormesh(x_top, y_top, fine_rgb.reshape(1, len(fine_rgb), 3), shading='flat')
    ax_fine.set_xlim(0, len(fine_rgb)); ax_fine.set_ylim(0, 1); ax_fine.set_xticks([]); ax_fine.set_yticks([])

    all_axes = [ax_broad, ax_fine]
    any_signature_track_drawn = False
    all_signature_colors = {}
    n_multiple_total = 0
    last_ax_ct = None
    last_ax_me = None
    row_cursor = 2
    n_blocks = len(built)
    for i, (big_me, big_ct, labels, target_labels, target_bold, b) in enumerate(built):
        ax_sig = fig.add_subplot(gs[row_cursor, 0])
        ax_dir = fig.add_subplot(gs[row_cursor, 1])
        ax_me = fig.add_subplot(gs[row_cursor, 2])
        ax_anno = fig.add_subplot(gs[row_cursor, 3])
        ax_ct = fig.add_subplot(gs[row_cursor, 4])

        is_last_block = (i == n_blocks - 1)
        show_sig_track = b.get("show_signature_track", False) and gene_to_signature_info is not None
        _draw_block_heatmaps(big_me, big_ct, labels, target_labels, ax_me, ax_ct,
                              cmap_me, vmin_me, vmax_me, cmap_ct, vmin_ct, vmax_ct,
                              show_right_labels=b.get("show_right_labels", True),
                              show_left_labels=False if show_sig_track else b.get("show_left_labels", True),
                              show_x_labels_me=b.get("show_x_labels_me", is_last_block),
                              show_x_labels_ct=b.get("show_x_labels_ct", is_last_block),
                              all_target_bold=target_bold)
        _draw_pdac_strip(ax_anno, big_me.index, gene_to_pdac)
        ax_anno.set_ylim(ax_me.get_ylim())

        if show_sig_track:
            sig_color_dict_block, n_multiple_block = _draw_signature_tracks(
                ax_sig, ax_dir, big_me.index, target_labels, target_bold, gene_to_signature_info)
            all_signature_colors.update(sig_color_dict_block)
            n_multiple_total += n_multiple_block
            ax_sig.set_ylim(ax_me.get_ylim())
            ax_dir.set_ylim(ax_me.get_ylim())
            all_axes.extend([ax_sig, ax_dir])
            any_signature_track_drawn = True
        else:
            ax_sig.axis('off')
            ax_dir.axis('off')

        all_axes.extend([ax_me, ax_anno, ax_ct])
        last_ax_ct = ax_ct
        last_ax_me = ax_me
        row_cursor += 2  

    for ax in all_axes:
        for spine in ax.spines.values():
            spine.set_linewidth(0.8); spine.set_edgecolor("black"); spine.set_visible(True)

    ax_cbar = last_ax_ct.inset_axes([1.05, 0.0, 0.025, 0.2], transform=last_ax_ct.transAxes)
    sm = ScalarMappable(cmap=cmap_ct, norm=plt.Normalize(vmin=vmin_ct, vmax=vmax_ct))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=ax_cbar)
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label("Normalized Expression", fontsize=12)

    if zscore_mass_spec:
        ax_cbar_me = last_ax_me.inset_axes([-0.22, 0.0, 0.025, 0.2], transform=last_ax_me.transAxes)
        sm_me = ScalarMappable(cmap=cmap_me, norm=plt.Normalize(vmin=vmin_me, vmax=vmax_me))
        sm_me.set_array([])
        cbar_me = fig.colorbar(sm_me, cax=ax_cbar_me)
        cbar_me.ax.tick_params(labelsize=10)
        cbar_me.ax.yaxis.set_label_position('left')
        cbar_me.ax.yaxis.set_ticks_position('right')
        cbar_me.set_label("Mass spec (z-score)", fontsize=12, labelpad=8)

    pdac_color_dict = {"Activated": "#3C5934", "Classical": "#7D8E9E", "Basal": "#A0BEC0", None: "#FFFFFF"}
    broad_handles = [Patch(facecolor=broad_color_dict[k], edgecolor="black", label=k) for k in broad_color_dict]
    fine_handles = [Patch(facecolor=fine_color_dict.get(c, "#CCCCCC"), edgecolor="black", label=c) for c in all_cols]
    pdac_handles = [Patch(facecolor=pdac_color_dict[k], edgecolor="black", label=str(k)) for k in ["Activated", "Classical", "Basal", None]]
    legend_y = 1 + (0.6 / total_height)
    leg1 = fig.legend(handles=broad_handles, title="Broad Cell Types", loc="upper left", bbox_to_anchor=(0.01, legend_y), frameon=True)
    leg2 = fig.legend(handles=fine_handles, title="Fine Cell Types", loc="upper center", bbox_to_anchor=(0.55, legend_y), ncol=4, frameon=True)
    leg3 = fig.legend(handles=pdac_handles, title="PDAC Class", loc="upper right", bbox_to_anchor=(0.99, legend_y), frameon=True)
    fig.add_artist(leg1); fig.add_artist(leg2); fig.add_artist(leg3)

    if any_signature_track_drawn:
        dir_handles = [Patch(facecolor=_DIRECTION_COLORS[k], edgecolor="black", label=k)
                       for k in ("Up", "Down", "Mixed")]
        leg4 = fig.legend(handles=dir_handles, title="Signature direction", loc="upper left",
                           bbox_to_anchor=(0.20, legend_y), frameon=True)
        fig.add_artist(leg4)

      
        _sig_rank = {name: i for i, name in enumerate(signature_order)} if signature_order else {}
        sig_handles = [Patch(facecolor=all_signature_colors[k], edgecolor="black", label=k)
                        for k in sorted(all_signature_colors, key=lambda k: _sig_rank.get(k, len(_sig_rank)))]
        if n_multiple_total > 0:
            sig_handles.append(Patch(facecolor=_MULTIPLE_SIGNATURE_COLOR, edgecolor="black",
                                      label=f"{_MULTIPLE_SIGNATURE_LABEL} (n={n_multiple_total})"))
        sig_ncol = 4
        sig_nrows = -(-len(sig_handles) // sig_ncol)  
        legend_y_sig = 1 + (0.6 + sig_nrows * 0.32 + 0.3) / total_height
        leg5 = fig.legend(handles=sig_handles, title="Signature", loc="upper left",
                           bbox_to_anchor=(0.01, legend_y_sig), ncol=sig_ncol, frameon=True, fontsize=7)
        fig.add_artist(leg5)

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return [bm for bm, *_ in built]


In [11]:
df_unimputed, df_filtered, meta_filtered = load_mass_spec()
final_genes = filter_candidate_genes(df_filtered, meta_filtered)
wilcoxon_df_sorted = run_wilcoxon(df_filtered, meta_filtered, final_genes)
mean_expr_ME = df_filtered.T.groupby(meta_filtered["ME_Type"]).mean().T
expanded_ms_genes = get_expanded_ms_genes(wilcoxon_df_sorted)

In [12]:
# Save the log2-transformed, minProb-imputed protein matrix (df_filtered) and
# its associated sample metadata (meta_filtered) to disk for Supplemental Tables
DF_FILTERED_CSV = OUT_DIR + "df_filtered_log2_imputed.csv"
META_FILTERED_CSV = OUT_DIR + "meta_filtered.csv"

df_filtered.to_csv(DF_FILTERED_CSV)
meta_filtered.to_csv(META_FILTERED_CSV)
print(f"Saved df_filtered {df_filtered.shape} -> {DF_FILTERED_CSV}")
print(f"Saved meta_filtered {meta_filtered.shape} -> {META_FILTERED_CSV}")

Saved df_filtered (6179, 297) -> output/df_filtered_log2_imputed.csv
Saved meta_filtered (297, 6) -> output/meta_filtered.csv


In [13]:
wilcoxon_df_sorted.loc[wilcoxon_df_sorted["FDR_within_ME"] < 0.05].to_csv(OUT_DIR + "Wilcoxon_significant_genes.csv", index=False)

In [14]:
print("meta_filtered ME_Type counts:")
print(meta_filtered["ME_Type"].value_counts())

meta_filtered ME_Type counts:
ME_Type
I      37
CCF    30
E      26
S      26
CH     24
M      23
NB     20
CM     17
CL     16
IS     15
B      15
H      15
SH     14
EX     12
D       7
Name: count, dtype: int64


In [15]:
# Curated therapeutic-target list
COMBINATION_SIGNATURES_TSV = INPUT_DIR + "COMBINATION_signatures.tsv"
combo_signatures = pd.read_csv(COMBINATION_SIGNATURES_TSV, sep="\t")

def build_gene_to_signature_map(df_combo):
    info = {}
    for _, row in df_combo.iterrows():
        sig_name = row["signature_name"]
        for col, direction in [("genes_required_high", "Up"), ("genes_required_low", "Down")]:
            val = row.get(col)
            if pd.isna(val) or not str(val).strip():
                continue
            for g in str(val).split(";"):
                g = g.strip().upper()
                if not g:
                    continue
                entry = info.setdefault(g, {"signatures": [], "directions": set()})
                entry["signatures"].append(sig_name)
                entry["directions"].add(direction)
    for g, entry in info.items():
        dirs = entry["directions"]
        entry["direction"] = dirs.pop() if len(dirs) == 1 else "Mixed"
    return info


gene_to_signature_info = build_gene_to_signature_map(combo_signatures)
genes_combination_signatures = sorted(gene_to_signature_info.keys())

signature_order = [
    # Chemo competence (drug processing/metabolism gates)
    "Gemcitabine competence",
    "Gemcitabine futility gate",
    "nab-paclitaxel delivery and target competence",
    "Fluoropyrimidine competence",
    "Irinotecan / nal-IRI competence",
    "Platinum / oxaliplatin competence",
    "FOLFIRINOX composite viability",
    "HRD / platinum-PARPi sensitivity",
    # Chemo resistance mechanisms
    "Gemcitabine resistance",
    "Intrinsic apoptosis blockade",
    "EMT master-regulator resistance program",
    "JAK/STAT3 inflammatory-survival axis",
    # KRAS / genomically-defined targeted therapy
    "KRAS inhibitor eligibility",
    "KRAS inhibitor adaptive-escape risk",
    "MTAP-deleted synthetic lethality",
    # ADC (linkers, payloads, antigens)
    "TOP1-inhibitor ADC payload viability",
    "Tubulin-payload ADC viability",
    "Non-cleavable maytansinoid viability",
    "Cleavable-linker processing competence",
    "Classical-programme surface antigen availability",
    "Basal-programme surface antigen availability",
    # Immunotherapy
    "T-cell-directed therapy floor",
    "Interferon axis integrity",
    "Suppressive myeloid niche",
    "CXCL12 T-cell exclusion axis",
    "Adenosine immunosuppression",
    "Macrophage checkpoint",
    # Stromal / physical barrier
    "Hyaluronan pressure barrier",
    "Stiff ECM immune-exclusion barrier",
    "FAP-addressable stroma",
    "TGF-beta immune exclusion",
]

# Sanity check: signature_order must exactly match the signature names present
# in COMBINATION_signatures.tsv 
_sheet_names = set(combo_signatures["signature_name"])
_missing_from_order = _sheet_names - set(signature_order)
_extra_in_order = set(signature_order) - _sheet_names
if _missing_from_order or _extra_in_order:
    raise ValueError(
        f"signature_order out of sync with COMBINATION_signatures.tsv. "
        f"Missing from signature_order: {_missing_from_order}. "
        f"Not in sheet: {_extra_in_order}."
    )

# Top heatmap block = genes that are part of a COMBINATION_signatures.tsv signature.
genes_therapeutic_targets = genes_combination_signatures
wilcoxon_annotated, nci_single_drugs = annotate_wilcoxon_with_targets(wilcoxon_df_sorted)

In [16]:
combined_gene_set = set(expanded_ms_genes) | set(genes_therapeutic_targets)
adata = rebuild_scrna(combined_gene_set, SC_OUT_H5AD)

/Users/Sibyl/miniconda3/envs/scanpy_spatialdata/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/Sibyl/miniconda3/envs/scanpy_spatialdata/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [17]:
tau = compute_tau(adata)
celltype_means = compute_celltype_means(adata)
tau_ms = compute_tau_ms(mean_expr_ME)
summary_table = build_summary_table(wilcoxon_annotated, mean_expr_ME, tau, adata)
subset, merged = filter_and_merge_subset(summary_table, mean_expr_ME, tau_ms, wilcoxon_annotated)
subset = filter_broad_group_specificity(subset, celltype_means)
subset.to_csv(OUT_DIR + "MS_SigGenes_scRNAseqTauFiltered_MSTausorted_minProb.csv")

/var/folders/wp/m71mqb5d54xdh2pz4bmqqtbh0000gp/T/ipykernel_8650/2636959826.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return adata.to_df().join(adata.obs['cell_type']).groupby('cell_type').mean().T


In [18]:
df_geneanno = pd.read_csv(
    INPUT_DIR + "MS_SigGenes_scRNAseqTauFiltered_MSTausorted_minProb_20260804.csv"
).set_index("gene")
color_map = pd.read_csv(INPUT_DIR + "Color_celltype_2026.csv", header=None, names=["celltype", "color"])
color_map["celltype"] = color_map["celltype"].str.strip()
fine_color_dict = dict(zip(color_map["celltype"], color_map["color"]))
df_PDACGenesets = pd.read_csv(PDAC_GENESETS_CSV)


In [19]:
from matplotlib.colors import LinearSegmentedColormap
USE_ZSCORE_FOR_MS = True

# Top block: COMBINATION_signatures.tsv genes
# (genes_therapeutic_targets was built in the curated-targets cell above).
subset_therapeutic_targets = build_subset_for_gene_list(
    genes_therapeutic_targets, mean_expr_ME, tau_ms
)

# spurious/indirect gene-drug interactions to exclude (fusion partners, likely
# gene-symbol confusion in DGIdb)
EXCLUDED_TARGET_GENES = {"KIF5B", "METTL1", "PIK3R4", "MAP2K3", "PIK3C2A", "EML4", "NPM1"}
subset_therapeutic_targets = subset_therapeutic_targets[
    ~subset_therapeutic_targets["gene"].isin(EXCLUDED_TARGET_GENES)
].copy()

blocks = [
    {
        "subset": subset_therapeutic_targets,
        "label_all_genes": True,
        "bold_all_genes": True,
        "show_right_labels": False,
        "row_height": (30 / len(subset)) * 3.5,
        "show_signature_track": True,
        "order_by_signature": True,
    },
    {
        "subset": subset,
        "height": 30,
    },
]

out_name = OUT_DIR + ("heatmap_therapeutic_over_wilcoxon_zscore.pdf" if USE_ZSCORE_FOR_MS else "heatmap_therapeutic_over_wilcoxon_minmax.pdf")


big_me_therapeutic, big_me_wilcoxon = plot_stacked_heatmaps(
    blocks, mean_expr_ME, celltype_means, df_geneanno, fine_color_dict,
    df_PDACGenesets, out_path=out_name,
    zscore_mass_spec=USE_ZSCORE_FOR_MS, zscore_vmax=2.0, 
    #zscore_cmap = LinearSegmentedColormap.from_list("fig4_rdbu", ["#F93026", "#FFFFFF", "#4358A7"], N=256),
    zscore_cmap = LinearSegmentedColormap.from_list("fig4_rdbu", ["#645EA4", "#FFFFFF", "#DA3B32"], N=256),
    gap_height=0.2,
    wilcoxon_annotated=wilcoxon_annotated,
    gene_to_signature_info=gene_to_signature_info,
    signature_order=signature_order,
)

